# Apples

### Import

In [1]:
import os, sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath('../scripts'))

from spread import SPREAD
from engine import ENGINE
from backtester import BACKTESTER
from tearsheet import TEARSHEET

### Data 

In [2]:
months = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512"
]

my_files = [
    [f"../data/processed/audusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/audusd_dukascopy_bid_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_bid_{m}.parquet" for m in months]
]

### SPREAD

In [3]:
builder = SPREAD(threshold=1000) 
df = builder.build(my_files)

print(df.head(5))

built 15455 rows
                                   Asset_A   Asset_B     Log_A     Log_B  \
timestamp                                                                  
2025-01-02 10:20:10.133000+00:00  0.621755  0.561855 -0.475209 -0.576511   
2025-01-02 10:25:55.560000+00:00  0.621930  0.561855 -0.474928 -0.576511   
2025-01-02 10:32:05.618000+00:00  0.621680  0.561780 -0.475330 -0.576645   
2025-01-02 10:34:14.918000+00:00  0.621880  0.561780 -0.475008 -0.576645   
2025-01-02 10:38:25.397000+00:00  0.621690  0.561840 -0.475314 -0.576538   

                                  Return_A  Return_B  
timestamp                                             
2025-01-02 10:20:10.133000+00:00  0.000265  0.000027  
2025-01-02 10:25:55.560000+00:00  0.000281  0.000000  
2025-01-02 10:32:05.618000+00:00 -0.000402 -0.000133  
2025-01-02 10:34:14.918000+00:00  0.000322  0.000000  
2025-01-02 10:38:25.397000+00:00 -0.000306  0.000107  


### ENGINE

In [4]:
from arch import arch_model # Ensure this is installed: pip install arch

print("⚙️ INITIATING WALK-FORWARD OOS PIPELINE...")

# 1. Prepare Date grouping
df['Date'] = df.index.date
unique_days = df['Date'].unique()

train_days = 30 
out_of_sample_results = []

print(f"Rolling Train Window: {train_days} days. Total Days: {len(unique_days)}")

# 2. The Walk-Forward Loop
for i in range(train_days, len(unique_days)):
    
    # Slice the chronological training and testing sets
    train_dates = unique_days[i - train_days : i]
    test_date = unique_days[i]
    
    train_df = df[df['Date'].isin(train_dates)].copy()
    test_df = df[df['Date'] == test_date].copy()
    
    # --- STEP A: FIT ON THE PAST ---
    engine = ENGINE(train_df)
    train_fitted = engine.fit_cointegration(z_window=1000)
    
    # We fit all three models here
    engine.fift_ar_reversion(lags=1)
    engine.fit_garch_vol(scaling=10000)
    engine.fit_markov_regimes(k_regimes=2)
    
    # --- STEP B: PREDICT THE FUTURE ---
    # Projects the frozen parameters onto the unseen test_df
    oos_predictions = engine.predict_oos(test_df, train_fitted, z_window=1000)
    out_of_sample_results.append(oos_predictions)
    
    if i % 10 == 0:
        print(f"✅ Processed up to {test_date} | Beta: {engine.beta:.4f} | GARCH Vol: {engine.forecasted_vol:.2f}")

# 3. Stitch results together
live_trading_data = pd.concat(out_of_sample_results)
print(f"\n🎉 OOS Dataset Built: {len(live_trading_data)} rows.")

⚙️ INITIATING WALK-FORWARD OOS PIPELINE...
Rolling Train Window: 30 days. Total Days: 259
Cointegration Fitted | Beta: 0.9434 | Alpha: 0.0690


NameError: name 'arch_model' is not defined

### BACKTEST

In [ ]:
print("🚀 RUNNING BACKTEST...")

# 1. Run the strategy simulation
bt = BACKTESTER(live_trading_data)
# We use the HMM Danger Probability as our main kill-switch
results_df = bt.run(
    entry_z=2.0, 
    exit_z=0.0, 
    danger_threshold=0.5, 
    fee_bps=0.5
)


### TEARSHEEET

In [ ]:
print("\n--- GENERATING FINANCIAL TEARSHEET ---")
ts = TEARSHEET(results_df)
ts.calculate_financials()

# 3. Plot the Equity Curve and Drawdowns
ts.plot_performance()